## Reto 11 Sistema de Gestión de Calificaciones

#### Autor

**Elena Carmina Mata Gonzalez Estudiante de Ciencia de Datos - Instituto Politécnico Nacional (IPN)**

#### Contexto del Problema

El departamento de Control Escolar del IPN necesita un **sistema para gestionar y analizar las calificaciones de estudiantes** de la carrera de Ciencia de Datos. El sistema debe permitir consultar datos, calcular promedios, identificar estudiantes en riesgo y generar reportes.

Tu tarea es crear un **gestor de calificaciones** usando Pandas DataFrames.

### Estructura de Datos

**Tabla: Estudiantes**

| Boleta | Nombre | Semestre | Carrera | Email |
| :--- | :--- | :--- | :--- | :--- |
| 2021630001 | Juan Pérez García | 4 | CD | juan@ipn.mx |
| 2021630002 | María López Ruiz | 4 | CD | maria@ipn.mx |

**Tabla: Calificaciones**

| Boleta | Materia_ID | Parcial_1 | Parcial_2 | Final |
| :--- | :--- | :--- | :--- | :--- |
| 2021630001 | MAT101 | 8.5 | 9.0 | 8.0 |
| 2021630001 | PROG201 | 9.0 | 8.5 | 9.5 |

**Tabla: Materias**

| Materia_ID | Nombre | Créditos | Semestre |
| :--- | :--- | :--- | :--- |
| MAT101 | Cálculo Diferencial | 8 | 1 |
| PROG201 | Programación Avanzada | 6 | 2 |

In [126]:
# Importaciones y configuración inicial
import pandas as pd
import numpy as np
from typing import Dict, List, Tuple, Optional
from datetime import datetime
import os
import warnings
warnings.filterwarnings('ignore')

### Parte 1. CARGA Y EXPLICACIÓN DE DATOS

**Fundamento Analítico: Ingesta y Calidad de Datos (Data Quality)**
El primer paso en cualquier pipeline de datos es la ingesta y validación. En esta sección simulamos la extracción de datos desde una base de datos relacional (simulando tablas de Estudiantes, Materias y Calificaciones). 

Antes de realizar cualquier cálculo, ejecutamos un proceso de perfilamiento de datos (Data Profiling) con la función `validar_datos()`. Esto nos permite detectar anomalías críticas (valores nulos o calificaciones fuera del dominio lógico de 0 a 10) garantizando que los análisis posteriores se construyan sobre información íntegra y confiable.

#### 1.1 Funció cargar_datos

In [127]:
def cargar_datos() -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Carga los datos de estudiantes, calificaciones y materias.
    
    Returns:
        Tupla con tres DataFrames: (df_estudiantes, df_calificaciones, df_materias)
    """
    # Datos de estudiantes
    estudiantes = pd.DataFrame({
        'boleta': ['2021630001', '2021630002', '2021630003', '2021630004', '2021630005',
                   '2022630001', '2022630002', '2022630003', '2022630004', '2022630005',
                   '2023630001', '2023630002', '2023630003', '2023630004', '2023630005'],
        'nombre': ['Juan Pérez García', 'María López Ruiz', 'Pedro Sánchez Torres',
                   'Ana Martínez Díaz', 'Luis Rodríguez Vega', 'Carmen Flores Luna',
                   'Roberto Díaz Mora', 'Laura Torres Silva', 'Diego Ramírez Cruz',
                   'Sofía Vargas Romo', 'Carlos Mendoza Ríos', 'Patricia Ortiz León',
                   'Miguel Ángel Castro', 'Fernanda Reyes Paz', 'Andrés Guzmán Villa'],
        'semestre': [4, 4, 4, 4, 4, 3, 3, 3, 3, 3, 2, 2, 2, 2, 2],
        'carrera': ['CD'] * 15,
        'email': ['juan.perez@ipn.mx', 'maria.lopez@ipn.mx', 'pedro.sanchez@ipn.mx',
                  'ana.martinez@ipn.mx', 'luis.rodriguez@ipn.mx', 'carmen.flores@ipn.mx',
                  'roberto.diaz@ipn.mx', 'laura.torres@ipn.mx', 'diego.ramirez@ipn.mx',
                  'sofia.vargas@ipn.mx', 'carlos.mendoza@ipn.mx', 'patricia.ortiz@ipn.mx',
                  'miguel.castro@ipn.mx', 'fernanda.reyes@ipn.mx', 'andres.guzman@ipn.mx']
    })
    
    # Datos de materias
    materias = pd.DataFrame({
        'materia_id': ['MAT101', 'MAT102', 'PROG101', 'PROG102', 'EST101', 'EST102', 'BD101'],
        'nombre': ['Cálculo Diferencial', 'Cálculo Integral', 'Programación I',
                   'Programación II', 'Probabilidad', 'Estadística Inferencial',
                   'Bases de Datos'],
        'creditos': [8, 8, 6, 6, 6, 6, 6],
        'semestre_materia': [1, 2, 1, 2, 2, 3, 3]
    })
    
    # Generar calificaciones (simuladas)
    np.random.seed(42)
    calificaciones_data = []
    
    for boleta in estudiantes['boleta']:
        semestre = estudiantes[estudiantes['boleta'] == boleta]['semestre'].values[0]
        materias_cursadas = materias[materias['semestre_materia'] <= semestre]['materia_id'].tolist()
        
        for materia in materias_cursadas:
            # Generar calificaciones aleatorias
            base = np.random.uniform(5, 10)
            p1 = round(min(10, max(0, base + np.random.normal(0, 1))), 1)
            p2 = round(min(10, max(0, base + np.random.normal(0, 1))), 1)
            final = round(min(10, max(0, base + np.random.normal(0, 0.5))), 1)
            
            # Algunos valores nulos aleatorios
            if np.random.random() < 0.05:
                p2 = np.nan
            
            calificaciones_data.append({
                'boleta': boleta,
                'materia_id': materia,
                'parcial_1': p1,
                'parcial_2': p2,
                'final': final
            })
    
    calificaciones = pd.DataFrame(calificaciones_data)
    
    return estudiantes, calificaciones, materias

#### 1.2 Función info_general

In [128]:
def info_general(df_estudiantes: pd.DataFrame, df_calificaciones: pd.DataFrame) -> Dict:
    """
    Genera información general del sistema.
    
    Utiliza los métodos de Pandas para obtener estadísticas del conjunto de datos.
    
    Args:
        df_estudiantes: DataFrame con información de estudiantes
        df_calificaciones: DataFrame con calificaciones
    
    Returns:
        Diccionario con información general del sistema
    """
    resultado = {
        "total_estudiantes": len(df_estudiantes),
        "total_registros_calif": len(df_calificaciones),
        "semestres": sorted(df_estudiantes['semestre'].unique().tolist()),
        "materias_con_registros": df_calificaciones['materia_id'].nunique()
    }
    
    return resultado

#### 1.3 Fución validar_datos

In [129]:
def validar_datos(df_calificaciones: pd.DataFrame) -> Dict:
    """
    Valida la integridad de los datos.
    
    Verifica:
    - Registros con valores nulos
    - Calificaciones fuera del rango 0-10
    - Validez general de los datos
    
    Args:
        df_calificaciones: DataFrame con calificaciones
    
    Returns:
        Diccionario con resultados de validación
    """
    # Columnas de calificaciones
    cols_calif = ['parcial_1', 'parcial_2', 'final']
    
    # Detectar registros con valores nulos
    registros_con_nulos = df_calificaciones[cols_calif].isna().any(axis=1).sum()
    
    # Detectar calificaciones fuera de rango (0-10)
    # Verificamos si hay valores menores a 0 o mayores a 10 en las columnas de calificaciones
    fuera_rango = 0
    for col in cols_calif:
        fuera_rango += ((df_calificaciones[col] < 0) | (df_calificaciones[col] > 10)).sum()
    
    # Los datos son válidos si no hay valores fuera de rango
    datos_validos = (fuera_rango == 0)
    
    resultado = {
        "registros_con_nulos": registros_con_nulos,
        "calificaciones_fuera_rango": fuera_rango,
        "datos_validos": datos_validos
    }
    
    return resultado

### Parte 2. CONSULTAS Y FILTROS

**Manipulación Relacional y Filtrado (Data Wrangling)**
En esta fase utilizamos el poder de Pandas para realizar consultas complejas sin necesidad de usar SQL. 

Destaca la construcción del Kardex, donde aplicamos la función `merge()` (equivalente a un JOIN en bases de datos) para cruzar las calificaciones de un estudiante con el catálogo maestro de materias. Además, implementamos indexación booleana (máscaras) para filtrar a la población estudiantil basándonos en rangos de rendimiento, lo que facilita la creación de cohortes de estudio.

#### 2.1 Función buscara_estudiante

In [130]:
def buscar_estudiante(df_estudiantes: pd.DataFrame, criterio: str, valor: str) -> pd.DataFrame:
    """
    Busca estudiantes por diferentes criterios.
    
    Args:
        df_estudiantes: DataFrame con información de estudiantes
        criterio: 'boleta', 'nombre', 'semestre'
        valor: valor a buscar (para nombre, búsqueda parcial)
    
    Returns:
        DataFrame con estudiantes encontrados
    """
    if criterio == 'boleta':
        # Búsqueda exacta por boleta
        return df_estudiantes[df_estudiantes['boleta'] == valor]
    
    elif criterio == 'nombre':
        # Búsqueda parcial por nombre (case-insensitive)
        return df_estudiantes[df_estudiantes['nombre'].str.contains(valor, case=False, na=False)]
    
    elif criterio == 'semestre':
        # Búsqueda por semestre (convertir a int)
        try:
            semestre = int(valor)
            return df_estudiantes[df_estudiantes['semestre'] == semestre]
        except ValueError:
            return pd.DataFrame()
    
    else:
        return pd.DataFrame()

#### 2.2 Función obtener_kardex

In [131]:
def obtener_kardex(boleta: str, df_estudiantes: pd.DataFrame,
                   df_calificaciones: pd.DataFrame, df_materias: pd.DataFrame) -> Dict:
    """
    Obtiene el kardex completo de un estudiante.
    
    Proceso:
    1. Obtiene los datos del estudiante
    2. Filtra sus calificaciones
    3. Une con información de materias
    4. Calcula promedios y estadísticas
    
    Args:
        boleta: Número de boleta del estudiante
        df_estudiantes: DataFrame con información de estudiantes
        df_calificaciones: DataFrame con calificaciones
        df_materias: DataFrame con información de materias
    
    Returns:
        Diccionario con el kardex completo
    """
    resultado = {
        "estudiante": None,
        "materias": None,
        "promedio_general": None,
        "creditos_cursados": None,
        "materias_aprobadas": None,
        "materias_reprobadas": None
    }
    
    # 1. Obtener datos del estudiante
    estudiante = df_estudiantes[df_estudiantes['boleta'] == boleta]
    if estudiante.empty:
        return resultado
    resultado["estudiante"] = estudiante.iloc[0].to_dict()
    
    # 2. Filtrar calificaciones del estudiante
    calif_estudiante = df_calificaciones[df_calificaciones['boleta'] == boleta].copy()
    
    if calif_estudiante.empty:
        resultado["materias"] = pd.DataFrame()
        return resultado
    
    # 3. Unir con nombres de materias
    calif_estudiante = calif_estudiante.merge(df_materias[['materia_id', 'nombre', 'creditos']], 
                                              on='materia_id')
    
    # 4. Calcular promedio por materia y agregar columna
    def calcular_promedio_fila(row):
        # Promedio de los tres parciales, ignorando NaN
        valores = [row['parcial_1'], row['parcial_2'], row['final']]
        valores_validos = [v for v in valores if pd.notna(v)]
        return round(sum(valores_validos) / len(valores_validos), 2) if valores_validos else None
    
    calif_estudiante['promedio'] = calif_estudiante.apply(calcular_promedio_fila, axis=1)
    calif_estudiante['estatus'] = calif_estudiante['promedio'].apply(
        lambda x: 'APROBADO' if pd.notna(x) and x >= 6 else 'REPROBADO' if pd.notna(x) else 'SIN CALIFICACIÓN'
    )
    
    # Guardar DataFrame de materias
    resultado["materias"] = calif_estudiante[['materia_id', 'nombre', 'creditos', 
                                               'parcial_1', 'parcial_2', 'final', 
                                               'promedio', 'estatus']]
    
    # 5. Calcular estadísticas
    aprobadas = calif_estudiante[calif_estudiante['estatus'] == 'APROBADO']
    reprobadas = calif_estudiante[calif_estudiante['estatus'] == 'REPROBADO']
    
    resultado["materias_aprobadas"] = len(aprobadas)
    resultado["materias_reprobadas"] = len(reprobadas)
    resultado["creditos_cursados"] = calif_estudiante['creditos'].sum()
    
    # Promedio general (promedio de promedios por materia)
    promedios_validos = calif_estudiante['promedio'].dropna()
    resultado["promedio_general"] = round(promedios_validos.mean(), 2) if not promedios_validos.empty else None
    
    return resultado

#### 2.3 Función filtrar_por_rendimiento

In [132]:
def filtrar_por_rendimiento(df_calificaciones: pd.DataFrame,
                            df_estudiantes: pd.DataFrame,
                            min_promedio: float = None,
                            max_promedio: float = None) -> pd.DataFrame:
    """
    Filtra estudiantes por rango de promedio.
    
    Proceso:
    1. Calcula el promedio de cada estudiante
    2. Aplica filtros según min_promedio y max_promedio
    3. Une con datos de estudiantes
    
    Args:
        df_calificaciones: DataFrame con calificaciones
        df_estudiantes: DataFrame con información de estudiantes
        min_promedio: Promedio mínimo para filtrar
        max_promedio: Promedio máximo para filtrar
    
    Returns:
        DataFrame con estudiantes y su promedio
    """
    # 1. Calcular promedio por estudiante
    def promedio_estudiante(boleta):
        calif = df_calificaciones[df_calificaciones['boleta'] == boleta]
        if calif.empty:
            return None
        
        # Calcular promedio de cada materia
        promedios = []
        for _, row in calif.iterrows():
            valores = [row['parcial_1'], row['parcial_2'], row['final']]
            valores_validos = [v for v in valores if pd.notna(v)]
            if valores_validos:
                promedios.append(sum(valores_validos) / len(valores_validos))
        
        return round(sum(promedios) / len(promedios), 2) if promedios else None
    
    # Crear DataFrame con promedios
    boletas_unicas = df_calificaciones['boleta'].unique()
    datos_promedios = []
    for boleta in boletas_unicas:
        prom = promedio_estudiante(boleta)
        if prom is not None:
            datos_promedios.append({'boleta': boleta, 'promedio': prom})
    
    df_promedios = pd.DataFrame(datos_promedios)
    
    if df_promedios.empty:
        return pd.DataFrame()
    
    # 2. Aplicar filtros según min_promedio y max_promedio
    mascara = pd.Series([True] * len(df_promedios))
    if min_promedio is not None:
        mascara = mascara & (df_promedios['promedio'] >= min_promedio)
    if max_promedio is not None:
        mascara = mascara & (df_promedios['promedio'] <= max_promedio)
    
    df_filtrado = df_promedios[mascara]
    
    if df_filtrado.empty:
        return pd.DataFrame()
    
    # 3. Unir con datos de estudiantes
    resultado = df_filtrado.merge(df_estudiantes, on='boleta')
    resultado = resultado[['boleta', 'nombre', 'semestre', 'carrera', 'promedio']]
    
    return resultado.sort_values('promedio', ascending=False)

### Parte 3. CÁLCULOS Y ESTADISTICAS

**Agregaciones y Estadísticas Descriptivas**
Esta sección transforma los datos crudos en métricas de negocio. Utilizamos el método `groupby()` combinado con `.agg()` para colapsar los registros individuales y encontrar patrones a nivel de cohorte (por semestre o por materia).

Calcular la "Tasa de Aprobación" y las medidas de tendencia central (media, máximos, mínimos) nos permite diagnosticar la salud académica de la institución y detectar "cuellos de botella" (materias con alto índice de reprobación) de forma completamente vectorizada.

#### 3.1 Función calcular_promedio_material

In [133]:
def calcular_promedio_materia(df_calificaciones: pd.DataFrame, materia_id: str) -> Dict:
    """
    Calcula estadísticas de una materia.
    
    Args:
        df_calificaciones: DataFrame con calificaciones
        materia_id: ID de la materia a analizar
    
    Returns:
        Diccionario con estadísticas de la materia
    """
    # Filtrar calificaciones de la materia
    df_materia = df_calificaciones[df_calificaciones['materia_id'] == materia_id].copy()
    
    resultado = {
        "materia": materia_id,
        "inscritos": len(df_materia),
        "promedio_parcial1": None,
        "promedio_parcial2": None,
        "promedio_final": None,
        "promedio_general": None,
        "tasa_aprobacion": None,
        "calificacion_maxima": None,
        "calificacion_minima": None
    }
    
    if df_materia.empty:
        return resultado
    
    # Calcular promedios de cada parcial
    resultado["promedio_parcial1"] = round(df_materia['parcial_1'].mean(), 2)
    resultado["promedio_parcial2"] = round(df_materia['parcial_2'].mean(), 2)
    resultado["promedio_final"] = round(df_materia['final'].mean(), 2)
    
    # Calcular promedio general (promedio de los 3 parciales por estudiante)
    promedios_estudiantes = []
    for _, row in df_materia.iterrows():
        valores = [row['parcial_1'], row['parcial_2'], row['final']]
        valores_validos = [v for v in valores if pd.notna(v)]
        if valores_validos:
            promedios_estudiantes.append(sum(valores_validos) / len(valores_validos))
    
    if promedios_estudiantes:
        resultado["promedio_general"] = round(sum(promedios_estudiantes) / len(promedios_estudiantes), 2)
        resultado["calificacion_maxima"] = round(max(promedios_estudiantes), 2)
        resultado["calificacion_minima"] = round(min(promedios_estudiantes), 2)
        
        # Tasa de aprobación (% de estudiantes con promedio >= 6)
        aprobados = sum(1 for p in promedios_estudiantes if p >= 6)
        resultado["tasa_aprobacion"] = round((aprobados / len(promedios_estudiantes)) * 100, 1)
    
    return resultado

#### 3.2 Función ranking_estudiantes

In [134]:
def ranking_estudiantes(df_calificaciones: pd.DataFrame,
                        df_estudiantes: pd.DataFrame,
                        top_n: int = 10) -> pd.DataFrame:
    """
    Genera ranking de mejores estudiantes por promedio.
    
    Proceso:
    1. Calcula el promedio de cada estudiante (todas sus materias)
    2. Ordena descendente por promedio
    3. Toma los top_n estudiantes
    4. Agrega datos del estudiante
    
    Args:
        df_calificaciones: DataFrame con calificaciones
        df_estudiantes: DataFrame con información de estudiantes
        top_n: Número de estudiantes a mostrar en el ranking
    
    Returns:
        DataFrame con ranking de estudiantes
    """
    # 1. Calcular promedio por estudiante
    def promedio_estudiante(boleta):
        calif = df_calificaciones[df_calificaciones['boleta'] == boleta]
        if calif.empty:
            return None
        
        promedios = []
        for _, row in calif.iterrows():
            valores = [row['parcial_1'], row['parcial_2'], row['final']]
            valores_validos = [v for v in valores if pd.notna(v)]
            if valores_validos:
                promedios.append(sum(valores_validos) / len(valores_validos))
        
        return round(sum(promedios) / len(promedios), 2) if promedios else None
    
    # Crear DataFrame con promedios
    boletas_unicas = df_calificaciones['boleta'].unique()
    datos_promedios = []
    for boleta in boletas_unicas:
        prom = promedio_estudiante(boleta)
        if prom is not None:
            datos_promedios.append({'boleta': boleta, 'promedio': prom})
    
    df_promedios = pd.DataFrame(datos_promedios)
    
    if df_promedios.empty:
        return pd.DataFrame()
    
    # 2. Ordenar descendente
    df_promedios = df_promedios.sort_values('promedio', ascending=False)
    
    # 3. Tomar top_n
    df_top = df_promedios.head(top_n).reset_index(drop=True)
    df_top.index = df_top.index + 1  # Posición 1-based
    
    # 4. Agregar datos del estudiante
    resultado = df_top.merge(df_estudiantes, on='boleta')
    resultado = resultado[['boleta', 'nombre', 'semestre', 'promedio']]
    resultado.index.name = 'Posición'
    
    return resultado

#### 3.3 Función estadisticas_por_semestre

In [135]:
def estadisticas_por_semestre(df_estudiantes: pd.DataFrame,
                              df_calificaciones: pd.DataFrame) -> pd.DataFrame:
    """
    Calcula estadísticas agrupadas por semestre.
    
    Proceso:
    1. Calcula el promedio de cada estudiante
    2. Une con información del semestre del estudiante
    3. Agrupa por semestre y calcula estadísticas
    
    Args:
        df_estudiantes: DataFrame con información de estudiantes
        df_calificaciones: DataFrame con calificaciones
    
    Returns:
        DataFrame con estadísticas por semestre
    """
    # 1. Calcular promedio por estudiante
    def promedio_estudiante(boleta):
        calif = df_calificaciones[df_calificaciones['boleta'] == boleta]
        if calif.empty:
            return None
        
        promedios = []
        for _, row in calif.iterrows():
            valores = [row['parcial_1'], row['parcial_2'], row['final']]
            valores_validos = [v for v in valores if pd.notna(v)]
            if valores_validos:
                promedios.append(sum(valores_validos) / len(valores_validos))
        
        return round(sum(promedios) / len(promedios), 2) if promedios else None
    
    # Crear DataFrame con promedios y semestres
    datos = []
    for _, estudiante in df_estudiantes.iterrows():
        boleta = estudiante['boleta']
        prom = promedio_estudiante(boleta)
        if prom is not None:
            datos.append({
                'boleta': boleta,
                'semestre': estudiante['semestre'],
                'promedio': prom
            })
    
    df_promedios = pd.DataFrame(datos)
    
    if df_promedios.empty:
        return pd.DataFrame()
    
    # 2. Agrupar por semestre y calcular estadísticas
    stats = df_promedios.groupby('semestre').agg({
        'boleta': 'count',  # Número de estudiantes
        'promedio': ['mean', 'min', 'max', 'std']  # Estadísticas de promedio
    })
    
    # Renombrar columnas
    stats.columns = ['Estudiantes', 'Promedio', 'Mínimo', 'Máximo', 'Desv Std']
    stats = stats.round(2)
    
    # Calcular tasa de aprobación por semestre
    def tasa_aprobacion_semestre(semestre):
        df_sem = df_promedios[df_promedios['semestre'] == semestre]
        if df_sem.empty:
            return 0
        aprobados = (df_sem['promedio'] >= 6).sum()
        return round((aprobados / len(df_sem)) * 100, 1)
    
    stats['Tasa Aprob.'] = stats.index.map(tasa_aprobacion_semestre)
    
    return stats

### Parte 4. IDENTIFICACIÓN DE RIESGOS Y REPORTE

**Lógica de Negocio y Generación de Insights (Reporting)**
El verdadero valor de la Ciencia de Datos radica en el soporte a la toma de decisiones. Aquí implementamos reglas de negocio para clasificar a los estudiantes en "Riesgo Académico" cruzando múltiples variables (promedio acumulado y conteo de materias reprobadas).

Finalmente, orquestamos todas las funciones anteriores para consolidar un "Reporte Ejecutivo" y automatizamos la exportación de resultados a formatos estándar de la industria (CSV y JSON), permitiendo la interoperabilidad con otros sistemas del IPN.

#### 4.1 Función identificar_estudiantes_riesgo

In [136]:
def identificar_estudiantes_riesgo(df_calificaciones: pd.DataFrame,
                                   df_estudiantes: pd.DataFrame,
                                   umbral_promedio: float = 7.0,
                                   max_reprobadas: int = 2) -> pd.DataFrame:
    """
    Identifica estudiantes en riesgo académico.
    
    Criterios de riesgo:
    - Promedio general < umbral_promedio
    - Más de max_reprobadas materias reprobadas
    
    Args:
        df_calificaciones: DataFrame con calificaciones
        df_estudiantes: DataFrame con información de estudiantes
        umbral_promedio: Promedio mínimo para considerar riesgo
        max_reprobadas: Número máximo de reprobadas permitido
    
    Returns:
        DataFrame con estudiantes en riesgo y motivo
    """
    # 1. Calcular promedio general y contar reprobadas por estudiante
    def analizar_estudiante(boleta):
        calif = df_calificaciones[df_calificaciones['boleta'] == boleta]
        if calif.empty:
            return None
        
        promedios_materias = []
        reprobadas = 0
        
        for _, row in calif.iterrows():
            valores = [row['parcial_1'], row['parcial_2'], row['final']]
            valores_validos = [v for v in valores if pd.notna(v)]
            if valores_validos:
                prom_materia = sum(valores_validos) / len(valores_validos)
                promedios_materias.append(prom_materia)
                if prom_materia < 6:
                    reprobadas += 1
        
        if not promedios_materias:
            return None
        
        promedio_general = sum(promedios_materias) / len(promedios_materias)
        
        return {
            'boleta': boleta,
            'promedio': round(promedio_general, 2),
            'reprobadas': reprobadas,
            'total_materias': len(promedios_materias)
        }
    
    # 2. Analizar todos los estudiantes
    boletas_unicas = df_calificaciones['boleta'].unique()
    datos_analisis = []
    for boleta in boletas_unicas:
        resultado = analizar_estudiante(boleta)
        if resultado:
            datos_analisis.append(resultado)
    
    df_analisis = pd.DataFrame(datos_analisis)
    
    if df_analisis.empty:
        return pd.DataFrame()
    
    # 3. Identificar estudiantes en riesgo
    # Bajo promedio: promedio < umbral_promedio
    # Muchas reprobadas: reprobadas > max_reprobadas
    mascara_bajo = df_analisis['promedio'] < umbral_promedio
    mascara_reprobadas = df_analisis['reprobadas'] > max_reprobadas
    
    df_riesgo = df_analisis[mascara_bajo | mascara_reprobadas].copy()
    
    if df_riesgo.empty:
        return pd.DataFrame()
    
    # 4. Agregar motivo
    def determinar_motivo(row):
        motivos = []
        if row['promedio'] < umbral_promedio:
            motivos.append("Bajo promedio")
        if row['reprobadas'] > max_reprobadas:
            motivos.append("Mat. reprob.")
        return ", ".join(motivos) if motivos else "Sin motivo específico"
    
    df_riesgo['motivo'] = df_riesgo.apply(determinar_motivo, axis=1)
    
    # 5. Unir con datos de estudiantes
    resultado = df_riesgo.merge(df_estudiantes, on='boleta')
    resultado = resultado[['boleta', 'nombre', 'semestre', 'promedio', 'reprobadas', 'motivo']]
    
    return resultado.sort_values('promedio')

#### 4.2 Función generar_reporte_academico

In [137]:
def generar_reporte_academico(df_estudiantes: pd.DataFrame,
                              df_calificaciones: pd.DataFrame,
                              df_materias: pd.DataFrame) -> Dict:
    """
    Genera reporte académico completo.
    
    Utiliza todas las funciones anteriores para construir un reporte integral.
    
    Args:
        df_estudiantes: DataFrame con información de estudiantes
        df_calificaciones: DataFrame con calificaciones
        df_materias: DataFrame con información de materias
    
    Returns:
        Diccionario con el reporte completo
    """
    # Calcular promedios de estudiantes para estadísticas globales
    def promedio_estudiante(boleta):
        calif = df_calificaciones[df_calificaciones['boleta'] == boleta]
        if calif.empty:
            return None
        
        promedios = []
        for _, row in calif.iterrows():
            valores = [row['parcial_1'], row['parcial_2'], row['final']]
            valores_validos = [v for v in valores if pd.notna(v)]
            if valores_validos:
                promedios.append(sum(valores_validos) / len(valores_validos))
        
        return round(sum(promedios) / len(promedios), 2) if promedios else None
    
    promedios_estudiantes = []
    for boleta in df_calificaciones['boleta'].unique():
        prom = promedio_estudiante(boleta)
        if prom is not None:
            promedios_estudiantes.append(prom)
    
    # Resumen general
    aprobados = sum(1 for p in promedios_estudiantes if p >= 6)
    total_estudiantes = len(promedios_estudiantes)
    
    resumen_general = {
        "total_estudiantes": total_estudiantes,
        "promedio_global": round(sum(promedios_estudiantes) / len(promedios_estudiantes), 2) if promedios_estudiantes else 0,
        "tasa_aprobacion": round((aprobados / total_estudiantes) * 100, 1) if total_estudiantes > 0 else 0
    }
    
    # Construir reporte
    reporte = {
        "resumen_general": resumen_general,
        "por_semestre": estadisticas_por_semestre(df_estudiantes, df_calificaciones),
        "por_materia": None,  # Puede ser extendido
        "mejores_estudiantes": ranking_estudiantes(df_calificaciones, df_estudiantes, top_n=5),
        "estudiantes_riesgo": identificar_estudiantes_riesgo(df_calificaciones, df_estudiantes),
        "fecha_generacion": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    }
    
    return reporte

#### 4.3 Función exportar_kardex

In [138]:
def exportar_kardex(boleta: str, kardex: Dict, formato: str = 'csv') -> str:
    """
    Exporta el kardex de un estudiante a archivo.
    
    Args:
        boleta: Número de boleta del estudiante
        kardex: Diccionario con el kardex del estudiante
        formato: 'csv' o 'json'
    
    Returns:
        Nombre del archivo generado
    """
    if kardex['estudiante'] is None or kardex['materias'] is None:
        print("No hay datos para exportar")
        return ""
    
    # Crear nombre de archivo
    nombre_estudiante = kardex['estudiante']['nombre'].replace(' ', '_')
    fecha = datetime.now().strftime("%Y%m%d")
    nombre_archivo = f"kardex_{boleta}_{nombre_estudiante}_{fecha}"
    
    # Preparar datos para exportar
    df_materias = kardex['materias'].copy()
    
    # Agregar información del estudiante al DataFrame
    df_materias['boleta'] = boleta
    df_materias['estudiante'] = kardex['estudiante']['nombre']
    df_materias['semestre'] = kardex['estudiante']['semestre']
    
    # Exportar según formato
    if formato.lower() == 'csv':
        archivo = f"{nombre_archivo}.csv"
        df_materias.to_csv(archivo, index=False, encoding='utf-8')
        print(f"Kardex exportado a: {archivo}")
        return archivo
    
    elif formato.lower() == 'json':
        archivo = f"{nombre_archivo}.json"
        df_materias.to_json(archivo, orient='records', force_ascii=False, indent=2)
        print(f"Kardex exportado a: {archivo}")
        return archivo
    
    else:
        print(f"Formato '{formato}' no soportado")
        return ""

### FUNCIONES DE VISUALIZACIÓN

In [139]:
def mostrar_kardex(kardex: Dict) -> None:
    """
    Muestra el kardex de forma legible.
    
    Args:
        kardex: Diccionario con el kardex del estudiante
    """
    if kardex['estudiante'] is None:
        print("Estudiante no encontrado")
        return
    
    est = kardex['estudiante']
    print("=" * 70)
    print("                         KARDEX ACADÉMICO")
    print("=" * 70)
    print(f"\nDATOS DEL ESTUDIANTE")
    print("-" * 40)
    print(f"Boleta: {est.get('boleta', 'N/A')}")
    print(f"Nombre: {est.get('nombre', 'N/A')}")
    print(f"Semestre: {est.get('semestre', 'N/A')}")
    print(f"Carrera: {est.get('carrera', 'N/A')}")
    print(f"Email: {est.get('email', 'N/A')}")
    
    print(f"\nCALIFICACIONES")
    print("-" * 70)
    if kardex['materias'] is not None and len(kardex['materias']) > 0:
        print(kardex['materias'].to_string(index=False))
    else:
        print("Sin calificaciones registradas")
    
    print(f"\n RESUMEN")
    print("-" * 40)
    print(f"Promedio General: {kardex.get('promedio_general', 0):.2f}")
    print(f"Créditos Cursados: {kardex.get('creditos_cursados', 0)}")
    print(f"Materias Aprobadas: {kardex.get('materias_aprobadas', 0)}")
    print(f"Materias Reprobadas: {kardex.get('materias_reprobadas', 0)}")
    print("=" * 70)

def mostrar_reporte(reporte: Dict) -> None:
    """
    Muestra el reporte académico completo.
    
    Args:
        reporte: Diccionario con el reporte académico
    """
    print("=" * 70)
    print("              REPORTE ACADÉMICO - CIENCIA DE DATOS")
    print(f"              Generado: {reporte['fecha_generacion']}")
    print("=" * 70)
    
    # Resumen general
    res = reporte.get('resumen_general', {})
    print(f"\nRESUMEN GENERAL")
    print("-" * 40)
    print(f"Total de estudiantes: {res.get('total_estudiantes', 'N/A')}")
    print(f"Promedio global: {res.get('promedio_global', 0):.2f}")
    print(f"Tasa de aprobación: {res.get('tasa_aprobacion', 0):.1f}%")
    
    # Por semestre
    if reporte.get('por_semestre') is not None and not reporte['por_semestre'].empty:
        print(f"\nESTADÍSTICAS POR SEMESTRE")
        print("-" * 40)
        print(reporte['por_semestre'].to_string())
    
    # Mejores estudiantes
    if reporte.get('mejores_estudiantes') is not None and not reporte['mejores_estudiantes'].empty:
        print(f"\nTOP 5 ESTUDIANTES")
        print("-" * 40)
        print(reporte['mejores_estudiantes'].head().to_string(index=True))
    
    # Estudiantes en riesgo
    riesgo = reporte.get('estudiantes_riesgo')
    if riesgo is not None and len(riesgo) > 0:
        print(f"\nESTUDIANTES EN RIESGO ({len(riesgo)})")
        print("-" * 40)
        print(riesgo.to_string(index=False))
    else:
        print(f"\nNo hay estudiantes en riesgo académico")
    
    print("\n" + "=" * 70)

### PRUEBA DE IMPLEMENTACIÓN

#### Cragar Datos

In [140]:
# Cargar datos
df_estudiantes, df_calificaciones, df_materias = cargar_datos()

print("DATOS CARGADOS")
print("=" * 50)
print(f"\nEstudiantes ({len(df_estudiantes)} registros):")
print(df_estudiantes.head())

print(f"\nCalificaciones ({len(df_calificaciones)} registros):")
print(df_calificaciones.head())

print(f"\nMaterias ({len(df_materias)} registros):")
print(df_materias)

DATOS CARGADOS

Estudiantes (15 registros):
       boleta                nombre  semestre carrera                  email
0  2021630001     Juan Pérez García         4      CD      juan.perez@ipn.mx
1  2021630002      María López Ruiz         4      CD     maria.lopez@ipn.mx
2  2021630003  Pedro Sánchez Torres         4      CD   pedro.sanchez@ipn.mx
3  2021630004     Ana Martínez Díaz         4      CD    ana.martinez@ipn.mx
4  2021630005   Luis Rodríguez Vega         4      CD  luis.rodriguez@ipn.mx

Calificaciones (95 registros):
       boleta materia_id  parcial_1  parcial_2  final
0  2021630001     MAT101        5.8        7.2    7.0
1  2021630001     MAT102        6.1        4.5    4.8
2  2021630001    PROG101        3.9        7.5    6.9
3  2021630001    PROG102        4.9        5.8    5.4
4  2021630001     EST101        8.6        6.4    6.1

Materias (7 registros):
  materia_id                   nombre  creditos  semestre_materia
0     MAT101      Cálculo Diferencial         8

#### Prueba de Información General

In [141]:
# Prueba de información general
print("\nINFORMACIÓN GENERAL")
print("=" * 50)
info = info_general(df_estudiantes, df_calificaciones)
for key, value in info.items():
    print(f"{key}: {value}")


INFORMACIÓN GENERAL
total_estudiantes: 15
total_registros_calif: 95
semestres: [2, 3, 4]
materias_con_registros: 7


#### Prueba de Validación de Datos

In [142]:
# Prueba de validación
print("\nVALIDACIÓN DE DATOS")
print("=" * 50)
validacion = validar_datos(df_calificaciones)
for key, value in validacion.items():
    print(f"{key}: {value}")


VALIDACIÓN DE DATOS
registros_con_nulos: 5
calificaciones_fuera_rango: 0
datos_validos: True


#### Prueba de Búsqueda de Estudiantes

In [143]:
# Prueba de búsqueda
print("\nBÚSQUEDA DE ESTUDIANTES")
print("=" * 50)

print("\n-- Buscar por nombre 'María' --")
resultado = buscar_estudiante(df_estudiantes, 'nombre', 'María')
print(resultado)

print("\n-- Buscar por semestre 3 --")
resultado = buscar_estudiante(df_estudiantes, 'semestre', '3')
print(resultado)

print("\n-- Buscar por boleta '2021630001' --")
resultado = buscar_estudiante(df_estudiantes, 'boleta', '2021630001')
print(resultado)


BÚSQUEDA DE ESTUDIANTES

-- Buscar por nombre 'María' --
       boleta            nombre  semestre carrera               email
1  2021630002  María López Ruiz         4      CD  maria.lopez@ipn.mx

-- Buscar por semestre 3 --
       boleta              nombre  semestre carrera                 email
5  2022630001  Carmen Flores Luna         3      CD  carmen.flores@ipn.mx
6  2022630002   Roberto Díaz Mora         3      CD   roberto.diaz@ipn.mx
7  2022630003  Laura Torres Silva         3      CD   laura.torres@ipn.mx
8  2022630004  Diego Ramírez Cruz         3      CD  diego.ramirez@ipn.mx
9  2022630005   Sofía Vargas Romo         3      CD   sofia.vargas@ipn.mx

-- Buscar por boleta '2021630001' --
       boleta             nombre  semestre carrera              email
0  2021630001  Juan Pérez García         4      CD  juan.perez@ipn.mx


#### Prueba de Kardex

In [144]:
# Prueba de kardex
print("\nKARDEX DE ESTUDIANTE")
print("=" * 50)
kardex = obtener_kardex('2021630001', df_estudiantes, df_calificaciones, df_materias)
mostrar_kardex(kardex)


KARDEX DE ESTUDIANTE
                         KARDEX ACADÉMICO

DATOS DEL ESTUDIANTE
----------------------------------------
Boleta: 2021630001
Nombre: Juan Pérez García
Semestre: 4
Carrera: CD
Email: juan.perez@ipn.mx

CALIFICACIONES
----------------------------------------------------------------------
materia_id                  nombre  creditos  parcial_1  parcial_2  final  promedio   estatus
    MAT101     Cálculo Diferencial         8        5.8        7.2    7.0      6.67  APROBADO
    MAT102        Cálculo Integral         8        6.1        4.5    4.8      5.13 REPROBADO
   PROG101          Programación I         6        3.9        7.5    6.9      6.10  APROBADO
   PROG102         Programación II         6        4.9        5.8    5.4      5.37 REPROBADO
    EST101            Probabilidad         6        8.6        6.4    6.1      7.03  APROBADO
    EST102 Estadística Inferencial         6        4.8        4.7    5.8      5.10 REPROBADO
     BD101          Bases de Datos

#### Prueba de Filtro por Rendimiento

In [145]:
# Prueba de filtro por rendimiento
print("\nFILTRO POR RENDIMIENTO")
print("=" * 50)

print("\n-- Estudiantes con promedio >= 8.0 --")
resultado = filtrar_por_rendimiento(df_calificaciones, df_estudiantes, min_promedio=8.0)
print(resultado)

print("\n-- Estudiantes con promedio entre 6.0 y 7.0 --")
resultado = filtrar_por_rendimiento(df_calificaciones, df_estudiantes, min_promedio=6.0, max_promedio=7.0)
print(resultado)


FILTRO POR RENDIMIENTO

-- Estudiantes con promedio >= 8.0 --
       boleta               nombre  semestre carrera  promedio
1  2023630003  Miguel Ángel Castro         2      CD      8.91
2  2023630004   Fernanda Reyes Paz         2      CD      8.59
0  2021630005  Luis Rodríguez Vega         4      CD      8.44
3  2023630005  Andrés Guzmán Villa         2      CD      8.43

-- Estudiantes con promedio entre 6.0 y 7.0 --
       boleta               nombre  semestre carrera  promedio
1  2021630002     María López Ruiz         4      CD      6.86
2  2023630002  Patricia Ortiz León         2      CD      6.46
0  2021630001    Juan Pérez García         4      CD      6.20


#### Prueba de promedio por Materia

In [146]:
# Prueba de promedio de materia
print("\nESTADÍSTICAS POR MATERIA")
print("=" * 50)

for materia in df_materias['materia_id'].unique():
    stats = calcular_promedio_materia(df_calificaciones, materia)
    print(f"\n{materia}:")
    print(f"  Inscritos: {stats['inscritos']}")
    print(f"  Promedio general: {stats['promedio_general']}")
    print(f"  Tasa de aprobación: {stats['tasa_aprobacion']}%")


ESTADÍSTICAS POR MATERIA

MAT101:
  Inscritos: 15
  Promedio general: 7.79
  Tasa de aprobación: 100.0%

MAT102:
  Inscritos: 15
  Promedio general: 7.45
  Tasa de aprobación: 86.7%

PROG101:
  Inscritos: 15
  Promedio general: 7.82
  Tasa de aprobación: 86.7%

PROG102:
  Inscritos: 15
  Promedio general: 7.52
  Tasa de aprobación: 86.7%

EST101:
  Inscritos: 15
  Promedio general: 8.14
  Tasa de aprobación: 93.3%

EST102:
  Inscritos: 10
  Promedio general: 7.03
  Tasa de aprobación: 80.0%

BD101:
  Inscritos: 10
  Promedio general: 7.46
  Tasa de aprobación: 90.0%


#### Prueba de Ranking

In [147]:
# Prueba de ranking
print("\nRANKING DE ESTUDIANTES")
print("=" * 50)
ranking = ranking_estudiantes(df_calificaciones, df_estudiantes, top_n=5)
print(ranking)


RANKING DE ESTUDIANTES
              boleta               nombre  semestre  promedio
Posición                                                     
0         2023630003  Miguel Ángel Castro         2      8.91
1         2023630004   Fernanda Reyes Paz         2      8.59
2         2021630005  Luis Rodríguez Vega         4      8.44
3         2023630005  Andrés Guzmán Villa         2      8.43
4         2022630003   Laura Torres Silva         3      7.96


#### Prueba de Estadisticas por Semestre

In [148]:
# Prueba de estadísticas por semestre
print("\nESTADÍSTICAS POR SEMESTRE")
print("=" * 50)
stats_semestre = estadisticas_por_semestre(df_estudiantes, df_calificaciones)
print(stats_semestre)


ESTADÍSTICAS POR SEMESTRE
          Estudiantes  Promedio  Mínimo  Máximo  Desv Std  Tasa Aprob.
semestre                                                              
2                   5      7.96    6.46    8.91      1.01        100.0
3                   5      7.75    7.41    7.96      0.21        100.0
4                   5      7.30    6.20    8.44      0.88        100.0


#### Prueba de Estudiantes en Riesgo

In [149]:
# Prueba de identificación de riesgo
print("\nESTUDIANTES EN RIESGO")
print("=" * 50)

print("\n-- Con criterios estándar (umbral=7.0, max_reprobadas=2) --")
riesgo = identificar_estudiantes_riesgo(df_calificaciones, df_estudiantes)
print(riesgo)

print("\n-- Con criterios más estrictos (umbral=8.0, max_reprobadas=1) --")
riesgo_estricto = identificar_estudiantes_riesgo(df_calificaciones, df_estudiantes, 
                                                  umbral_promedio=8.0, max_reprobadas=1)
print(riesgo_estricto)


ESTUDIANTES EN RIESGO

-- Con criterios estándar (umbral=7.0, max_reprobadas=2) --
       boleta               nombre  semestre  promedio  reprobadas  \
0  2021630001    Juan Pérez García         4      6.20           3   
2  2023630002  Patricia Ortiz León         2      6.46           1   
1  2021630002     María López Ruiz         4      6.86           0   

                        motivo  
0  Bajo promedio, Mat. reprob.  
2                Bajo promedio  
1                Bajo promedio  

-- Con criterios más estrictos (umbral=8.0, max_reprobadas=1) --
        boleta                nombre  semestre  promedio  reprobadas  \
0   2021630001     Juan Pérez García         4      6.20           3   
10  2023630002   Patricia Ortiz León         2      6.46           1   
1   2021630002      María López Ruiz         4      6.86           0   
2   2021630003  Pedro Sánchez Torres         4      7.11           2   
9   2023630001   Carlos Mendoza Ríos         2      7.40           1   
7   2

#### Prueba de Reporte Completo

In [150]:
# Prueba de reporte completo
print("\nREPORTE ACADÉMICO COMPLETO")
print("=" * 50)
reporte = generar_reporte_academico(df_estudiantes, df_calificaciones, df_materias)
mostrar_reporte(reporte)


REPORTE ACADÉMICO COMPLETO
              REPORTE ACADÉMICO - CIENCIA DE DATOS
              Generado: 2026-06-23 09:23:10

RESUMEN GENERAL
----------------------------------------
Total de estudiantes: 15
Promedio global: 7.67
Tasa de aprobación: 100.0%

ESTADÍSTICAS POR SEMESTRE
----------------------------------------
          Estudiantes  Promedio  Mínimo  Máximo  Desv Std  Tasa Aprob.
semestre                                                              
2                   5      7.96    6.46    8.91      1.01        100.0
3                   5      7.75    7.41    7.96      0.21        100.0
4                   5      7.30    6.20    8.44      0.88        100.0

TOP 5 ESTUDIANTES
----------------------------------------
              boleta               nombre  semestre  promedio
Posición                                                     
0         2023630003  Miguel Ángel Castro         2      8.91
1         2023630004   Fernanda Reyes Paz         2      8.59
2         2021

#### Prueba de Exportación en Kardex

In [151]:
# Prueba de exportación de kardex
print("\nEXPORTACIÓN DE KARDEX")
print("=" * 50)

# Exportar a CSV
archivo_csv = exportar_kardex('2021630001', kardex, 'csv')

# Exportar a JSON
archivo_json = exportar_kardex('2021630001', kardex, 'json')

# Mostrar contenido del archivo CSV exportado
if archivo_csv and os.path.exists(archivo_csv):
    print(f"\nContenido del archivo CSV exportado:")
    df_exportado = pd.read_csv(archivo_csv)
    print(df_exportado)


EXPORTACIÓN DE KARDEX
Kardex exportado a: kardex_2021630001_Juan_Pérez_García_20260623.csv
Kardex exportado a: kardex_2021630001_Juan_Pérez_García_20260623.json

Contenido del archivo CSV exportado:
  materia_id                   nombre  creditos  parcial_1  parcial_2  final  \
0     MAT101      Cálculo Diferencial         8        5.8        7.2    7.0   
1     MAT102         Cálculo Integral         8        6.1        4.5    4.8   
2    PROG101           Programación I         6        3.9        7.5    6.9   
3    PROG102          Programación II         6        4.9        5.8    5.4   
4     EST101             Probabilidad         6        8.6        6.4    6.1   
5     EST102  Estadística Inferencial         6        4.8        4.7    5.8   
6      BD101           Bases de Datos         6        7.4        8.3    8.2   

   promedio    estatus      boleta         estudiante  semestre  
0      6.67   APROBADO  2021630001  Juan Pérez García         4  
1      5.13  REPROBADO  202

#### Limpieza de Archivos

In [152]:
# Limpieza de archivos de exportación
print("\nLIMPIANDO ARCHIVOS...")
for archivo in ['kardex_2021630001_Juan_Pérez_García_*.csv', 'kardex_2021630001_Juan_Pérez_García_*.json']:
    import glob
    for f in glob.glob(archivo):
        #os.remove(f) # se comenta para evitar la eliminación accidental durante pruebas
        print(f"Eliminado: {f}")
print("Archivos temporales eliminados.")


LIMPIANDO ARCHIVOS...
Eliminado: kardex_2021630001_Juan_Pérez_García_20260623.csv
Eliminado: kardex_2021630001_Juan_Pérez_García_20260623.json
Archivos temporales eliminados.


### FUNCIONES BONUS

**Analítica Predictiva y Comparativa Avanzada**
Yendo más allá del análisis descriptivo, implementamos una heurística de alerta temprana. El *Predictor de Riesgo* escanea la trayectoria intra-semestral de los estudiantes (comparando el Parcial 1 contra el Final) para identificar tendencias de deterioro académico. 

Este enfoque proactivo permite a Control Escolar intervenir antes de que el estudiante acumule un historial irrecuperable.

#### Predictor de Riesgo

In [153]:
def predecir_riesgo_proximo_semestre(df_calificaciones: pd.DataFrame,
                                     df_estudiantes: pd.DataFrame) -> pd.DataFrame:
    """
    Predice estudiantes que podrían estar en riesgo el próximo semestre
    basándose en tendencias de calificaciones.
    
    Criterios:
    - Tendencia decreciente en parciales
    - Calificación final menor que parcial 1
    
    Args:
        df_calificaciones: DataFrame con calificaciones
        df_estudiantes: DataFrame con información de estudiantes
    
    Returns:
        DataFrame con estudiantes en riesgo potencial
    """
    predicciones = []
    
    for boleta in df_calificaciones['boleta'].unique():
        calif = df_calificaciones[df_calificaciones['boleta'] == boleta]
        if calif.empty:
            continue
        
        estudiante = df_estudiantes[df_estudiantes['boleta'] == boleta]
        if estudiante.empty:
            continue
        
        # Analizar tendencias en cada materia
        riesgo_materias = 0
        for _, row in calif.iterrows():
            if pd.isna(row['parcial_1']) or pd.isna(row['final']):
                continue
            
            # Tendencia decreciente: final < parcial_1
            if row['final'] < row['parcial_1']:
                riesgo_materias += 1
        
        # Si tiene más de 2 materias con tendencia decreciente, está en riesgo
        if riesgo_materias >= 2:
            predicciones.append({
                'boleta': boleta,
                'nombre': estudiante.iloc[0]['nombre'],
                'semestre': estudiante.iloc[0]['semestre'],
                'materias_con_tendencia_decreciente': riesgo_materias,
                'motivo': 'Tendencia decreciente en calificaciones'
            })
    
    if not predicciones:
        return pd.DataFrame()
    
    return pd.DataFrame(predicciones)

#### Comparador de Estudiantes

In [154]:
def comparar_estudiantes(boleta1: str, boleta2: str,
                         df_calificaciones: pd.DataFrame,
                         df_estudiantes: pd.DataFrame,
                         df_materias: pd.DataFrame) -> Dict:
    """
    Compara el rendimiento de dos estudiantes.
    
    Args:
        boleta1: Boleta del primer estudiante
        boleta2: Boleta del segundo estudiante
        df_calificaciones: DataFrame con calificaciones
        df_estudiantes: DataFrame con información de estudiantes
        df_materias: DataFrame con información de materias
    
    Returns:
        Diccionario con la comparación
    """
    kardex1 = obtener_kardex(boleta1, df_estudiantes, df_calificaciones, df_materias)
    kardex2 = obtener_kardex(boleta2, df_estudiantes, df_calificaciones, df_materias)
    
    resultado = {
        'estudiante1': kardex1,
        'estudiante2': kardex2,
        'comparacion': {}
    }
    
    if kardex1['estudiante'] is None or kardex2['estudiante'] is None:
        resultado['comparacion'] = {'error': 'Uno de los estudiantes no existe'}
        return resultado
    
    # Calcular diferencia de promedios
    prom1 = kardex1['promedio_general'] or 0
    prom2 = kardex2['promedio_general'] or 0
    
    resultado['comparacion'] = {
        'diferencia_promedio': round(prom1 - prom2, 2),
        'mejor_promedio': kardex1['estudiante']['nombre'] if prom1 > prom2 else kardex2['estudiante']['nombre'],
        'diferencia_creditos': kardex1['creditos_cursados'] - kardex2['creditos_cursados'],
        'diferencia_aprobadas': kardex1['materias_aprobadas'] - kardex2['materias_aprobadas']
    }
    
    return resultado

#### Prueba de Funciones Bonus

In [155]:
# Prueba de predictor de riesgo
print("\nPREDICTOR DE RIESGO PARA PRÓXIMO SEMESTRE")
print("=" * 50)
prediccion = predecir_riesgo_proximo_semestre(df_calificaciones, df_estudiantes)
if not prediccion.empty:
    print(prediccion)
else:
    print("No se encontraron estudiantes en riesgo potencial")


PREDICTOR DE RIESGO PARA PRÓXIMO SEMESTRE
        boleta                nombre  semestre  \
0   2021630001     Juan Pérez García         4   
1   2021630002      María López Ruiz         4   
2   2021630003  Pedro Sánchez Torres         4   
3   2021630004     Ana Martínez Díaz         4   
4   2021630005   Luis Rodríguez Vega         4   
5   2022630001    Carmen Flores Luna         3   
6   2022630002     Roberto Díaz Mora         3   
7   2022630004    Diego Ramírez Cruz         3   
8   2022630005     Sofía Vargas Romo         3   
9   2023630001   Carlos Mendoza Ríos         2   
10  2023630002   Patricia Ortiz León         2   
11  2023630003   Miguel Ángel Castro         2   
12  2023630004    Fernanda Reyes Paz         2   
13  2023630005   Andrés Guzmán Villa         2   

    materias_con_tendencia_decreciente  \
0                                    2   
1                                    4   
2                                    4   
3                                    3

In [156]:
# Prueba de comparador de estudiantes
print("\nCOMPARADOR DE ESTUDIANTES")
print("=" * 50)

boleta1 = '2021630001'  # Juan Pérez García
boleta2 = '2021630003'  # Pedro Sánchez Torres

comparacion = comparar_estudiantes(boleta1, boleta2, df_calificaciones, df_estudiantes, df_materias)

print(f"Comparación entre {comparacion['estudiante1']['estudiante']['nombre']} y {comparacion['estudiante2']['estudiante']['nombre']}:")
print(f"  Diferencia de promedios: {comparacion['comparacion']['diferencia_promedio']}")
print(f"  Mejor promedio: {comparacion['comparacion']['mejor_promedio']}")
print(f"  Diferencia de créditos: {comparacion['comparacion']['diferencia_creditos']}")
print(f"  Diferencia de aprobadas: {comparacion['comparacion']['diferencia_aprobadas']}")


COMPARADOR DE ESTUDIANTES
Comparación entre Juan Pérez García y Pedro Sánchez Torres:
  Diferencia de promedios: -0.91
  Mejor promedio: Pedro Sánchez Torres
  Diferencia de créditos: 0
  Diferencia de aprobadas: -1


#### CONCLUSIONES

In [157]:
print("\n" + "=" * 70)
print("                    RESUMEN DEL SISTEMA")
print("=" * 70)

print(f"\nDATOS GENERALES")
print("-" * 40)
print(f"Total de estudiantes: {len(df_estudiantes)}")
print(f"Total de materias: {len(df_materias)}")
print(f"Total de registros de calificaciones: {len(df_calificaciones)}")
print(f"Semestres: {sorted(df_estudiantes['semestre'].unique())}")

# Estadísticas de rendimiento
promedios = []
for boleta in df_calificaciones['boleta'].unique():
    calif = df_calificaciones[df_calificaciones['boleta'] == boleta]
    prom_materias = []
    for _, row in calif.iterrows():
        valores = [row['parcial_1'], row['parcial_2'], row['final']]
        valores_validos = [v for v in valores if pd.notna(v)]
        if valores_validos:
            prom_materias.append(sum(valores_validos) / len(valores_validos))
    if prom_materias:
        promedios.append(sum(prom_materias) / len(prom_materias))

if promedios:
    print(f"\nRENDIMIENTO GENERAL")
    print("-" * 40)
    print(f"Promedio general: {sum(promedios)/len(promedios):.2f}")
    print(f"Promedio máximo: {max(promedios):.2f}")
    print(f"Promedio mínimo: {min(promedios):.2f}")
    aprobados = sum(1 for p in promedios if p >= 6)
    print(f"Tasa de aprobación: {(aprobados/len(promedios))*100:.1f}%")

print("\n" + "=" * 70)
print("SISTEMA DE GESTIÓN DE CALIFICACIONES COMPLETADO")
print("=" * 70)


                    RESUMEN DEL SISTEMA

DATOS GENERALES
----------------------------------------
Total de estudiantes: 15
Total de materias: 7
Total de registros de calificaciones: 95
Semestres: [np.int64(2), np.int64(3), np.int64(4)]

RENDIMIENTO GENERAL
----------------------------------------
Promedio general: 7.67
Promedio máximo: 8.91
Promedio mínimo: 6.20
Tasa de aprobación: 100.0%

SISTEMA DE GESTIÓN DE CALIFICACIONES COMPLETADO


### Conclusión General del Proyecto

El desarrollo de este Sistema de Gestión de Calificaciones demuestra cómo la manipulación eficiente de datos puede transformar registros operativos en **inteligencia institucional**. 

A través de la integración y análisis de las tablas de Estudiantes, Materias y Calificaciones, el sistema logró:
1. **Automatizar el Seguimiento Académico:** Sustituyendo consultas manuales por Kardex dinámicos y métricas de cohortes (promedios por semestre y tasa de aprobación general del 100% en la muestra).
2. **Implementar Reglas de Negocio Proactivas:** La función de detección de riesgo estándar identificó a 3 estudiantes vulnerables basándose en su rendimiento acumulado, permitiendo focalizar esfuerzos de tutoría.
3. **Analítica Predictiva (Bonus):** El modelo de alerta temprana fue capaz de escanear la trayectoria intra-semestral, detectando patrones de deterioro en el rendimiento de los parciales antes de que se conviertan en materias reprobadas definitivas.

La exportación automatizada de estos insights a formatos estándar (CSV/JSON) garantiza que el departamento de Control Escolar pueda integrar estos hallazgos en sus tableros de control diarios.